# MLP-VaR Softplus: validacion 2010-2014 y test 2015-2024

Notebook experimental. Replica la logica del MLP-VaR original y cambia solo la salida para imponer VaR positivo mediante `Softplus(linear_output)`.

No sobrescribe CSVs antiguos. Genera archivos nuevos `nn_softplus_validation_w*.csv`, `nn_softplus_test_w*.csv` y resumenes Softplus.

In [ ]:
from pathlib import Path
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import chi2
from tqdm import tqdm

DATA_DIR = Path("../data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_VALUES = [1, 5, 10, 15, 20, 25]
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42

torch.set_num_threads(1)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

## Carga de datos y comprobacion de escala

In [ ]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta la columna target"

feature_cols = [c for c in dataset.columns if c != "target"]
assert len(feature_cols) == 22, f"Se esperaban 22 features, hay {len(feature_cols)}"

print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features:", len(feature_cols))
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

# Check de plausibilidad: datos en tanto por uno, no en porcentaje.
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece estar en porcentaje o tiene escala inesperada"
assert dataset[feature_cols].abs().quantile(0.999).max() < 0.5, "Alguna feature parece tener escala inesperada"

In [ ]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))

## Metricas y tests de backtesting

In [ ]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def exceedance_stats(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, I = hit_series(real_losses, var_preds)
    T = len(I)
    x = int(I.sum())
    p = 1 - alpha
    rv = x / T if T > 0 else np.nan
    return {
        "T": T,
        "Expected Viol.": p * T,
        "Violations": x,
        "Violation Rate": rv,
        "VR Gap": abs(rv - p) if T > 0 else np.nan,
        "Coverage Bias": rv - p if T > 0 else np.nan,
    }


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def mean_var_level(var_preds):
    return float(np.nanmean(np.asarray(var_preds, dtype=float)))


def p_value_mean(p_uc, p_cc, p_dq):
    vals = [v for v in [p_uc, p_cc, p_dq] if pd.notnull(v)]
    return float(np.mean(vals)) if vals else np.nan

In [ ]:
def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}

    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1

    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)

    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        LRcc, p_cc = np.nan, np.nan
    else:
        LRcc = LRuc + LRind
        p_cc = 1 - chi2.cdf(LRcc, df=2)
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": p_cc}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    p_dq = 1 - chi2.cdf(DQ, df=X.shape[1])
    return {"DQ": DQ, "p_dq": p_dq}

In [ ]:
def evaluate_var_model(real_losses, var_preds, alpha=0.99, sig=0.05):
    exc = exceedance_stats(real_losses, var_preds, alpha=alpha)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    var_preds = np.asarray(var_preds, dtype=float)

    row = {
        "T": exc["T"],
        "Expected Viol.": exc["Expected Viol."],
        "Violations": exc["Violations"],
        "Violation Rate": exc["Violation Rate"],
        "VR Gap": exc["VR Gap"],
        "Coverage Bias": exc["Coverage Bias"],
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": mean_var_level(var_preds),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "n_negative_var": int(np.sum(var_preds < 0)),
        "LRuc": kup["LRuc"],
        "p_UC": kup["p_uc"],
        "LRcc": cc["LRcc"],
        "p_CC": cc["p_cc"],
        "DQ": dq["DQ"],
        "p_DQ": dq["p_dq"],
    }
    row["UC Test"] = "PASS" if row["p_UC"] > sig else "FAIL"
    row["CC Test"] = "PASS" if row["p_CC"] > sig else "FAIL"
    row["DQ Test"] = "PASS" if row["p_DQ"] > sig else "FAIL"
    row["p_Mean"] = p_value_mean(row["p_UC"], row["p_CC"], row["p_DQ"])
    row["Valid(CC&DQ)"] = "YES" if (row["CC Test"] == "PASS" and row["DQ Test"] == "PASS") else "NO"
    return row


def add_2020_metrics(row, df, var_col="VaR_pred"):
    df_2020 = df.loc["2020"] if "2020" in df.index.strftime("%Y").unique() else pd.DataFrame()
    if len(df_2020) == 0:
        row["Mean VaR 2020"] = np.nan
        row["Max VaR 2020"] = np.nan
    else:
        row["Mean VaR 2020"] = float(np.nanmean(df_2020[var_col].values))
        row["Max VaR 2020"] = float(np.nanmax(df_2020[var_col].values))
    return row


SUMMARY_COLUMNS = [
    "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
    "Mean VaR 2020", "Max VaR 2020", "n_negative_var",
    "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
]

## Modelo MLP-VaR Softplus

In [ ]:
def inverse_softplus(y):
    # Bias raw que produce Softplus(raw) ~= y. Mantiene el arranque inicial en VaR ~= 2%.
    return math.log(math.expm1(y))


def var_loss(y_true, y_pred, q=0.99, w=20.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplus(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def train_one_model(X_train, y_train, d_in, seed, w_loss, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]

    model = MLPVaRSoftplus(d_in=d_in)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)

    best_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()

        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion rolling y guardado

In [ ]:
def run_one_softplus(w_val, eval_start, eval_end, out_csv):
    eval_dates = dataset.loc[pd.Timestamp(eval_start):pd.Timestamp(eval_end)].index
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    muX, sdX = None, None

    for i, current_date in enumerate(tqdm(eval_dates, desc=f"Softplus w={w_val}", dynamic_ncols=True)):
        if i % RETRAIN_EVERY == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)

            if len(train_df) < 250:
                if current_model is None:
                    continue
            else:
                X_train = train_df[feature_cols].values.astype(np.float32)
                y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
                muX = X_train.mean(axis=0, keepdims=True)
                sdX = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - muX) / sdX
                current_model = train_one_model(X_train, y_train, d_in=X_train.shape[1], seed=SEED, w_loss=float(w_val), alpha=ALPHA)

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

        var_preds.append(float(pred))
        real_targets.append(float(y_test.reshape(-1)[0]))
        dates.append(current_date)

    out_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    out_df = out_df.loc[pd.Timestamp(eval_start):pd.Timestamp(eval_end)].dropna().copy()
    out_df["viol"] = (out_df["loss_real"] > out_df["VaR_pred"]).astype(int)
    out_df["year"] = out_df.index.year
    out_df.to_csv(out_csv)
    return out_df


def summarize_prediction_df(w_val, df):
    row = evaluate_var_model(df["loss_real"].values, df["VaR_pred"].values, alpha=ALPHA, sig=SIG)
    row["w"] = w_val
    row = add_2020_metrics(row, df, var_col="VaR_pred")
    return row


def run_softplus_experiment(label, eval_start, eval_end, file_prefix, summary_path):
    rows = []
    outputs = {}
    for w_val in W_VALUES:
        out_csv = DATA_DIR / f"{file_prefix}_w{w_val}.csv"
        print(f"\n{'=' * 80}\n{label} | w={w_val} | output={out_csv}\n{'=' * 80}")
        df_pred = run_one_softplus(w_val, eval_start, eval_end, out_csv)
        plausibility_report(df_pred)
        row = summarize_prediction_df(w_val, df_pred)
        rows.append(row)
        outputs[w_val] = df_pred

    summary = pd.DataFrame(rows)[SUMMARY_COLUMNS]
    summary.to_csv(summary_path, index=False)
    print(f"\nGuardado resumen: {summary_path}")
    display(summary)
    return summary, outputs

## Parte A - Validacion 2010-2014

Esta celda entrena y evalua `w = 1, 5, 10, 15, 20, 25` en el periodo de validacion. Puede tardar bastante porque replica el rolling diario.

In [ ]:
softplus_validation_summary, softplus_validation_predictions = run_softplus_experiment(
    label="VALIDATION 2010-2014",
    eval_start="2010-01-01",
    eval_end="2014-12-31",
    file_prefix="nn_softplus_validation",
    summary_path=DATA_DIR / "softplus_validation_summary.csv",
)

### Lectura de validacion

Tras ejecutar la tabla anterior, revisar:

- Valores de `w` con `Valid(CC&DQ) = YES`.
- Si `n_negative_var` es cero para todos los `w`, Softplus elimina completamente VaR negativos.
- Comparar `Mean VaR Level`, `Max VaR Level` y `Tick Loss` frente a la tabla original de validacion.
- Buscar una region estable: varios `w` consecutivos con CC y DQ en PASS, tasas de violacion cercanas al 1% y niveles de VaR no extremos.
- No seleccionar automaticamente un ganador aqui; usar esta tabla como filtro de validacion.

## Parte B - Evaluacion final 2015-2024

Esta celda repite el mismo experimento en el periodo final de test.

In [ ]:
softplus_test_summary, softplus_test_predictions = run_softplus_experiment(
    label="TEST 2015-2024",
    eval_start="2015-01-01",
    eval_end="2024-12-31",
    file_prefix="nn_softplus_test",
    summary_path=DATA_DIR / "softplus_test_summary.csv",
)

### Lectura de test

Tras ejecutar la tabla anterior, revisar:

- Valores de `w` con `Valid(CC&DQ) = YES`.
- Si `n_negative_var` es cero para todos los `w`.
- Si Softplus reduce `Max VaR Level`, `Mean VaR Level`, `Mean VaR 2020` y `Max VaR 2020` frente al MLP original.
- Si la `Tick Loss` mejora o empeora frente al MLP original.
- Si la cobertura sigue siendo razonable: `Violation Rate` cerca de 0.01 y p-valores no rechazados.
- Para el TFG, priorizar una configuracion que no produzca VaR negativos, no tenga picos economicamente indefendibles y mantenga validez CC y DQ.

## Comparacion con MLP original por valor de w

In [ ]:
def load_original_predictions(period, w_val):
    if period == "validation":
        path = DATA_DIR / f"nn_val_predictions_{w_val}.csv"
    elif period == "test":
        path = DATA_DIR / f"nn_var_predictions_{w_val}.csv"
    else:
        raise ValueError(period)
    return pd.read_csv(path, index_col=0, parse_dates=True)


def comparison_rows_for_period(period, original_label, softplus_label, softplus_summary):
    rows = []
    for w_val in W_VALUES:
        original_df = load_original_predictions(period, w_val)
        original_row = summarize_prediction_df(w_val, original_df)
        original_row["Model"] = original_label
        rows.append(original_row)

        softplus_row = softplus_summary.loc[softplus_summary["w"] == w_val].iloc[0].to_dict()
        softplus_row["Model"] = softplus_label
        rows.append(softplus_row)

    cmp = pd.DataFrame(rows)
    cols = [
        "Model", "w", "Violations", "Violation Rate", "p_UC", "p_CC", "p_DQ",
        "Tick Loss", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
        "n_negative_var", "Valid(CC&DQ)",
    ]
    return cmp[cols]


comparison_validation = comparison_rows_for_period(
    period="validation",
    original_label="MLP original validation",
    softplus_label="MLP Softplus validation",
    softplus_summary=softplus_validation_summary,
)

comparison_test = comparison_rows_for_period(
    period="test",
    original_label="MLP original test",
    softplus_label="MLP Softplus test",
    softplus_summary=softplus_test_summary,
)

display(comparison_validation)
display(comparison_test)

In [ ]:
def answer_softplus_questions(original_summary, softplus_summary, period_label):
    joined = original_summary.merge(softplus_summary, on="w", suffixes=(" original", " softplus"))
    out = pd.DataFrame({
        "w": joined["w"],
        "negativos original": joined["n_negative_var original"],
        "negativos softplus": joined["n_negative_var softplus"],
        "reduce Max VaR": joined["Max VaR Level softplus"] < joined["Max VaR Level original"],
        "reduce Mean VaR": joined["Mean VaR Level softplus"] < joined["Mean VaR Level original"],
        "mejora Tick Loss": joined["Tick Loss softplus"] < joined["Tick Loss original"],
        "valid original": joined["Valid(CC&DQ) original"],
        "valid softplus": joined["Valid(CC&DQ) softplus"],
    })
    print(f"\nPreguntas Softplus - {period_label}")
    display(out)
    return out


def original_summary_for_period(period):
    rows = []
    for w_val in W_VALUES:
        df = load_original_predictions(period, w_val)
        rows.append(summarize_prediction_df(w_val, df))
    return pd.DataFrame(rows)[SUMMARY_COLUMNS]


original_validation_summary = original_summary_for_period("validation")
original_test_summary = original_summary_for_period("test")

softplus_questions_validation = answer_softplus_questions(original_validation_summary, softplus_validation_summary, "validation")
softplus_questions_test = answer_softplus_questions(original_test_summary, softplus_test_summary, "test")

## Informe final a partir de las tablas

Usar las tablas anteriores para responder explicitamente:

1. `n_negative_var`: si Softplus es cero en todos los `w`, elimina VaR negativos.
2. `reduce Max VaR`: indica si reduce picos extremos por cada `w`.
3. `reduce Mean VaR`: indica si reduce el nivel medio por cada `w`.
4. `Valid(CC&DQ)`: indica si mantiene validez estadistica.
5. `mejora Tick Loss`: indica si mejora la perdida cuantifica.
6. Defendibilidad TFG: sera mayor si elimina negativos, reduce extremos y conserva cobertura razonable sin empeorar demasiado `Tick Loss`.